# cuda-matmul on Colab / Kaggle

1. **Colab:** Runtime → Change runtime type → **T4 GPU**. **Kaggle:** Settings → Accelerator → **GPU T4 x2** (or P100), and turn Internet on.
2. Set your GitHub username below and run the cells top to bottom.
3. Download `results.zip` at the end and commit its contents to `results/` in your repo.

In [ ]:
GITHUB_USER = "Navneesh004"
REPO = "cuda-matmul"
!nvidia-smi
!nvcc --version

In [ ]:
import os
if not os.path.exists(REPO):
    !git clone https://github.com/{GITHUB_USER}/{REPO}.git
%cd {REPO}
!make
!./matmul --info
!./matmul --list

## Quick check: one kernel, one size

In [ ]:
!./matmul --kernel 3 --n 2048

## Correctness at awkward sizes (1, 7, 1000, 1023, ...)

In [ ]:
!bash scripts/test.sh

## Tune kernel 7's tile size, then benchmark everything (takes a few minutes)

In [ ]:
!bash scripts/tune.sh
!bash scripts/bench.sh

## Profile with Nsight Compute

If this prints `ERR_NVGPUCTRPERM`, the hardware counters are blocked on this machine.
Skip this cell (the roofline then uses theoretical intensities) or try Kaggle / a local GPU.

In [ ]:
!bash scripts/profile.sh

## Charts

In [ ]:
!pip -q install -r requirements.txt
!python3 scripts/plot.py
!python3 scripts/roofline.py
from IPython.display import Image, Markdown, display
for f in ["results/timing.png", "results/sizes.png", "results/roofline.png"]:
    if os.path.exists(f):
        display(Image(f))
display(Markdown(open("results/summary.md").read()))
if os.path.exists("results/ncu_summary.md"):
    display(Markdown(open("results/ncu_summary.md").read()))

## Download the results and commit them to your repo

In [ ]:
!zip -qr results.zip results -x 'results/ncu/*'
try:
    from google.colab import files
    files.download("results.zip")
except ImportError:
    print("On Kaggle: grab results.zip from the Output panel on the right.")